<p style="align: center;"><img src="https://static.tildacdn.com/tild6636-3531-4239-b465-376364646465/Deep_Learning_School.png" width="400"></p>

# Глубокое обучение. Часть 2
# Домашнее задание по теме "Механизм внимания"

Это домашнее задание проходит в формате peer-review. Это означает, что его будут проверять ваши однокурсники. Поэтому пишите разборчивый код, добавляйте комментарии и пишите выводы после проделанной работы.

В этом задании вы будете решать задачу классификации математических задач по темам (многоклассовая классификация) с помощью Transformer.

В качестве датасета возьмем датасет математических задач по разным темам. Нам необходим следующий файл:

[Файл с классами](https://docs.google.com/spreadsheets/d/13YIbphbWc62sfa-bCh8MLQWKizaXbQK9/edit?usp=drive_link&ouid=104379615679964018037&rtpof=true&sd=true)

**Hint:** не перезаписывайте модели, которые вы получите на каждом из этапов этого дз. Они ещё понадобятся.

### Задание 1 (2 балла)

Напишите кастомный класс для модели трансформера для задачи классификации, использующей в качествке backbone какую-то из моделей huggingface.

Т.е. конструктор класса должен принимать на вход название модели и подгружать её из huggingface, а затем использовать в качестве backbone (достаточно возможности использовать в качестве backbone те модели, которые упомянуты в последующих пунктах)

In [ ]:
!pip install evaluate

In [ ]:
import pandas as pd
from tqdm.auto import tqdm
from typing import Union, List

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from datasets import Dataset, DatasetDict
import transformers
import evaluate

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
!gdown 13YIbphbWc62sfa-bCh8MLQWKizaXbQK9

Downloading...
From: https://drive.google.com/uc?id=13YIbphbWc62sfa-bCh8MLQWKizaXbQK9
To: /content/data_problems_translated.xlsx
100% 526k/526k [00:00<00:00, 8.36MB/s]


In [ ]:
data = pd.read_excel("/content/data_problems_translated.xlsx", index_col=0)

In [ ]:
texts = data["problem_text"].astype(str).values
topics = data["topic"].astype(str).values

label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(topics)

num_classes = len(label_encoder.classes_)
print(num_classes)

df = pd.DataFrame({
    "text": texts,
    "label": labels
})

train_df, test_df = train_test_split(
    df,
    test_size=0.1,
    shuffle=True,
    random_state=42,
    stratify=df["label"]
)

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True))
})

7


In [ ]:
tokenizer = transformers.AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2")

In [ ]:
def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["text"],
        max_length=128,
        truncation=True,
        padding=False
    )

    model_inputs["labels"] = examples["label"]

    return model_inputs

In [ ]:
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["text", "label"]
)

Map:   0%|          | 0/4745 [00:00<?, ? examples/s]

Map:   0%|          | 0/528 [00:00<?, ? examples/s]

In [ ]:
class TransformerClassificationModel(nn.Module):
    def __init__(self, base_transformer_model: Union[str, nn.Module], num_classes: int):
        super().__init__()

        if isinstance(base_transformer_model, str):
            self.backbone = transformers.AutoModel.from_pretrained(
                base_transformer_model,
                attn_implementation="eager"
            )
        else:
            self.backbone = base_transformer_model

        hidden_size = self.backbone.config.hidden_size

        self.classifier = nn.Linear(hidden_size, num_classes)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls_embedding)

        result = {"logits": logits}

        if labels is not None:
            loss = self.loss_fn(logits, labels)
            result["loss"] = loss

        return result

### Задание 2 (1 балл)

Напишите функцию заморозки backbone у модели (если необходимо, возвращайте из функции модель)

In [ ]:
def freeze_backbone_function(model: TransformerClassificationModel):
    for param in model.backbone.parameters():
        param.requires_grad = False
    return model

### Задание 3 (2 балла)

Напишите функцию, которая будет использована для тренировки (дообучения) трансформера (TransformerClassificationModel). Функция должна поддерживать обучение с замороженным и размороженным backbone.

In [ ]:
import copy

def train_transformer(transformer_model, freeze_backbone=True):
    model = copy.deepcopy(transformer_model)

    if freeze_backbone:
        model = freeze_backbone_function(model)
    else:
        for param in model.backbone.parameters():
            param.requires_grad = True

    data_collator = transformers.DataCollatorWithPadding(tokenizer=tokenizer)

    training_args = transformers.TrainingArguments(
        output_dir="./results",
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=2e-5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        weight_decay=0.01,
        num_train_epochs=2,
        report_to="none"
    )

    trainer = transformers.Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["test"],
        processing_class=tokenizer,
        data_collator=data_collator,
    )

    trainer.train()

    return model

### Задание 4 (1 балл)

Проверьте вашу функцию из предыдущего пункта, дообучив двумя способами
*cointegrated/rubert-tiny2* из huggingface.

In [ ]:
rubert_tiny_transformer_model = TransformerClassificationModel(
    "cointegrated/rubert-tiny2",
    num_classes=num_classes
)

rubert_tiny_finetuned_with_freezed_backbone = train_transformer(
    rubert_tiny_transformer_model,
    freeze_backbone=True
)

rubert_tiny_transformer_model = TransformerClassificationModel(
    "cointegrated/rubert-tiny2",
    num_classes=num_classes
)

rubert_tiny_full_finetuned = train_transformer(
    rubert_tiny_transformer_model,
    freeze_backbone=False
)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super()

Epoch,Training Loss,Validation Loss
1,1.779614,1.694671
2,1.639168,1.627320


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss
1,1.174163,1.098748
2,1.008474,1.045295


### Задание 5 (1 балл)

Обучите *tbs17/MathBert* (с замороженным backbone и без заморозки), проанализируйте результаты. Сравните скоры с первым заданием. Получилось лучше или нет? Почему?

In [ ]:
tokenizer = transformers.AutoTokenizer.from_pretrained("tbs17/MathBert")

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["text"],
        max_length=128,
        truncation=True,
        padding=False
    )

    model_inputs["labels"] = examples["label"]

    return model_inputs

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["text", "label"]
)

data_collator = transformers.DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
mathbert_transformer_model = TransformerClassificationModel(
    base_transformer_model="tbs17/MathBert",
    num_classes=num_classes
)

mathbert_finetuned_with_freezed_backbone = train_transformer(
    mathbert_transformer_model,
    freeze_backbone=True
)

mathbert_transformer_model = TransformerClassificationModel(
    base_transformer_model="tbs17/MathBert",
    num_classes=num_classes
)

mathbert_full_finetuned = train_transformer(
    mathbert_transformer_model,
    freeze_backbone=False
)

MathBERT показал лучшие результаты при замороженном backbone по сравнению с rubert-tiny2, так как его validation loss ниже: 1.5325 против 1.5889. Это ожидаемо, потому что MathBERT предобучен на математических текстах и его готовые эмбеддинги лучше подходят для задач с математической лексикой.

При полном дообучении обе модели дали значительно лучший результат, чем при заморозке backbone. Это объясняется тем, что в режиме full fine-tuning обучается не только классификационная голова, но и сам transformer-backbone, поэтому модель адаптируется под конкретный датасет.

Однако после полного дообучения rubert-tiny2 оказался немного лучше по финальному validation loss: 1.0460 против 1.0730 у MathBERT. При этом у MathBERT лучший validation loss был после первой эпохи — 1.0475, а на второй эпохе validation loss вырос до 1.0730, хотя training loss снизился с 1.1173 до 0.7805. Это может говорить о начале переобучения MathBERT.

Таким образом, MathBERT лучше работает как замороженный feature extractor, но при полном дообучении rubert-tiny2 показал немного лучший финальный результат. В целом полное дообучение оказалось лучше замороженного backbone для обеих моделей.

### Задание 6 (1 балл)

Напишите функцию для отрисовки карт внимания первого слоя для моделей из задания

In [ ]:
def draw_first_layer_attention_maps(attention_head_ids: List, text: str, model: TransformerClassificationModel):
    model.eval()

    device = next(model.parameters()).device

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model.backbone(
            **inputs,
            output_attentions=True
        )

    attentions = outputs.attentions

    first_layer_attention = attentions[0]

    tokens = tokenizer.convert_ids_to_tokens(
        inputs["input_ids"][0].detach().cpu().tolist()
    )

    first_layer_attention = first_layer_attention[0].detach().cpu()

    for head_id in attention_head_ids:
        attention_map = first_layer_attention[head_id]

        plt.figure(figsize=(10, 8))
        plt.imshow(attention_map)

        plt.xticks(
            ticks=range(len(tokens)),
            labels=tokens,
            rotation=90
        )

        plt.yticks(
            ticks=range(len(tokens)),
            labels=tokens
        )

        plt.title(f"First layer attention map, head {head_id}")
        plt.colorbar()
        plt.tight_layout()
        plt.show()

In [ ]:
text = test_df["text"].iloc[0]

draw_first_layer_attention_maps(
    attention_head_ids=[0, 1, 2],
    text=text,
    model=rubert_tiny_finetuned_with_freezed_backbone
)

NameError: name 'draw_first_layer_attention_maps' is not defined

### Задание 7 (1 балл)

Проведите инференс для всех моделей **ДО ДООБУЧЕНИЯ** на 2-3 текстах из датасета. Посмотрите на головы Attention первого слоя в каждой модели на выбранных текстах (отрисуйте их отдельно).

Попробуйте их проинтерпретировать. Какие связи улавливают карты внимания? (если в модели много голов Attention, то проинтерпретируйте наиболее интересные)

In [ ]:
rubert_tiny_before_training = TransformerClassificationModel(
    base_transformer_model="cointegrated/rubert-tiny2",
    num_classes=num_classes
)

mathbert_before_training = TransformerClassificationModel(
    base_transformer_model="tbs17/MathBert",
    num_classes=num_classes
)

texts_for_attention = test_df["text"].iloc[:3].tolist()

tokenizer = transformers.AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2")

for text in texts_for_attention:
    print(text)
    draw_first_layer_attention_maps(
        attention_head_ids=[0, 1],
        text=text,
        model=rubert_tiny_before_training
    )

tokenizer = transformers.AutoTokenizer.from_pretrained("tbs17/MathBert")

for text in texts_for_attention:
    print(text)
    draw_first_layer_attention_maps(
        attention_head_ids=[0, 1],
        text=text,
        model=mathbert_before_training
    )

До дообучения карты внимания выглядят более общими: головы часто обращают внимание на специальные токены [CLS], [SEP], знаки пунктуации и соседние токены. Явной связи с конкретным математическим классом задачи пока не видно, потому что модель ещё не подстраивалась под датасет классификации.

### Задание 8 (1 балл)

Сделайте то же самое для дообученных моделей. Изменились ли карты внимания и связи, которые они улавливают? Почему?

In [ ]:
tokenizer = transformers.AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2")

for text in texts_for_attention:
    print(text)

    draw_first_layer_attention_maps(
        attention_head_ids=[0, 1],
        text=text,
        model=rubert_tiny_finetuned_with_freezed_backbone
    )

    draw_first_layer_attention_maps(
        attention_head_ids=[0, 1],
        text=text,
        model=rubert_tiny_full_finetuned
    )

После дообучения у модели с замороженным backbone карты внимания почти не меняются, потому что веса трансформера не обучались. Менялась только классификационная голова.

У модели без заморозки карты внимания могут измениться сильнее, потому что обучались все слои backbone. В этом случае отдельные головы могут начать сильнее выделять токены, связанные с темой задачи: числа, математические термины, имена объектов, ключевые слова условия.